# AIGC 5005 · Module 3 · Part 3
## Modules, packages, and imports

Companion notebook to Part 3 of the Module 3 learning content. Every example from that part is here, ready to run, with comments explaining what to notice.

**Estimated time:** 30 minutes

**How to use it:** read each explanation, run the cell beneath it, and compare the output with what you expected *before* you ran it. The prediction is where the learning happens. Change values and run cells again; you cannot break anything.

**Kernel:** select the kernel from your `module03` conda environment (top right in VS Code or Jupyter). To make one: `conda create -n module03 python=3.11 ipykernel`, then `conda activate module03`. If VS Code offers to install `ipykernel`, accept.

## 0. Set up a scratch folder

This notebook **writes real files to disk** so you can import them, exactly as you would with your own package. Everything goes inside a folder called `scratch_part3` next to this notebook. The last cell deletes it.

In [ ]:
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

SCRATCH = Path("scratch_part3").resolve()
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)          # start clean every time you re-run the notebook
(SCRATCH / "demo_pkg").mkdir(parents=True)
print("Working in:", SCRATCH)

## 1. Build a small package

Layout we are creating:

```
scratch_part3/
    main.py
    demo_pkg/
        __init__.py
        catalogue.py
        helpers.py
```

A **module** is one `.py` file. A **package** is a folder of modules with an `__init__.py`.

In [ ]:
(SCRATCH / "demo_pkg" / "helpers.py").write_text('''"""Small helper functions. Supporting code, not part of the public API."""


def format_title(title):
    return title.strip().title()
''')

(SCRATCH / "demo_pkg" / "catalogue.py").write_text('''"""Core classes for the catalogue package."""
from .helpers import format_title   # relative import: only works when imported AS PART OF the package


class Book:
    def __init__(self, title, copies=1):
        self.title = format_title(title)
        self.copies = copies


class Member:
    def __init__(self, name):
        self.name = name
''')

(SCRATCH / "demo_pkg" / "__init__.py").write_text('''"""Public interface of the demo_pkg package."""
# Re-export the names users should rely on, so they can write: from demo_pkg import Book
from .catalogue import Book, Member

__all__ = ["Book", "Member"]
''')

(SCRATCH / "main.py").write_text('''from demo_pkg import Book, Member           # names exposed by __init__.py
from demo_pkg.helpers import format_title   # an explicit import from a submodule

b = Book(format_title("  fluent python  "))
m = Member("Aisha")
print(b.title, m.name)
''')

for path in sorted(SCRATCH.rglob("*.py")):
    print(path.relative_to(SCRATCH))

## 2. Import it

Python finds packages by searching the folders listed in `sys.path`. We add our scratch folder to that list so `import demo_pkg` can find it. (Your real projects will use an editable install instead; see the worked-example notebook.)

**Predict first:** `__init__.py` re-exports `Book` and `Member`. Which import lines below will work?

In [ ]:
sys.path.insert(0, str(SCRATCH))
importlib.invalidate_caches()
for name in [n for n in sys.modules if n.startswith("demo_pkg")]:
    del sys.modules[name]           # forget any earlier import so re-running this cell is safe

from demo_pkg import Book, Member           # works: __init__.py re-exports these names
from demo_pkg.helpers import format_title   # works: helpers.py is reachable by its full path

b = Book(format_title("  fluent python  "))
m = Member("Aisha")
print(b.title, m.name)                      # Fluent Python Aisha

### Where did each name actually come from?

Every function and class remembers the module it was defined in. The public name `demo_pkg.Book` is really defined in `demo_pkg.catalogue`.

In [ ]:
import demo_pkg

print(demo_pkg.__all__)              # ['Book', 'Member']
print(Book.__module__)               # demo_pkg.catalogue  (defined here...)
print(demo_pkg.Book is Book)         # True                (...but re-exported at the top level)
print(Path(demo_pkg.__file__).name)  # __init__.py

### Imports are cached

The first `import` runs the module's code and stores the result in `sys.modules`. Later imports reuse it. This is why editing a module and re-importing it in the same session often seems to do nothing.

In [ ]:
print("demo_pkg" in sys.modules)               # True
print("demo_pkg.catalogue" in sys.modules)     # True
print(sys.modules["demo_pkg.catalogue"] is importlib.import_module("demo_pkg.catalogue"))   # True: same object

## 3. Run it the way a script would

`main.py` sits **outside** the `demo_pkg` folder, so it can import the package. We run it in a separate Python process, just as you would from a terminal.

In [ ]:
result = subprocess.run([sys.executable, "main.py"], cwd=SCRATCH, capture_output=True, text=True)
print(result.stdout, end="")        # Fluent Python Aisha
print("exit code:", result.returncode)

## 4. The classic mistake: running a package file directly

`catalogue.py` uses a **relative import** (`from .helpers import ...`). That only works when Python knows which package the file belongs to. Running it directly as a script hides that, so Python fails.

**Predict first:** what will the last line of the error say?

In [ ]:
result = subprocess.run([sys.executable, "demo_pkg/catalogue.py"], cwd=SCRATCH, capture_output=True, text=True)
print(result.stderr.strip().splitlines()[-1])
# ImportError: attempted relative import with no known parent package

### The fix: run it as part of the package

`python -m package.module` tells Python to treat the file as part of its package, so the relative import resolves. (You may see a harmless `RuntimeWarning`: `__init__.py` already imported `catalogue`, and `-m` then runs it again. It does not stop the program.)

In [ ]:
result = subprocess.run([sys.executable, "-m", "demo_pkg.catalogue"], cwd=SCRATCH, capture_output=True, text=True)
print("exit code:", result.returncode)         # 0: no ImportError this time
print("stderr  :", "RuntimeWarning" in result.stderr)   # True on most versions: a warning, not an error

### Try it

Build your own package using this notebook's pattern: a `shapes` package with `circle.py` and `square.py` (one class each), an `__init__.py` that re-exports both classes, and a `main.py` outside the package. Use the cell below to write the files, then run `main.py` with `subprocess` as above.

In [ ]:
# Your shapes package here.

## Clean up

Remove the scratch folder and take it off the import path.

In [ ]:
sys.path.remove(str(SCRATCH))
for name in [n for n in sys.modules if n.startswith("demo_pkg")]:
    del sys.modules[name]
shutil.rmtree(SCRATCH)
print("Scratch folder removed:", not SCRATCH.exists())